In [11]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_fi = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/fi_2007_2015_bn.csv")

In [3]:
df_fi = df_fi.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_fi

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,DI,GE,CPI,RQ,VAA,EUPC
0,8.57,5.64,6.31,4.54,17.0,0.43,30.67,25.553435,20.768314,4.865721,...,72.435714,-0.04,-825.28,-2010.0,9.03,1.97,94.0,1.55,1.46,6.033898e+06
1,6.79,5.84,6.43,4.52,17.0,0.47,29.57,25.248516,21.566205,0.316321,...,72.435714,0.04,522.16,-8272.0,9.25,2.04,90.0,1.61,1.46,9.016756e+06
2,-3.48,5.89,6.33,4.63,18.0,0.48,25.66,21.474124,24.069112,-8.514605,...,71.500000,-0.03,-1655.62,-3519.0,9.03,2.23,89.0,1.81,1.47,3.447843e+07
3,4.90,5.86,6.15,4.44,18.0,0.46,26.19,22.607323,23.685735,2.697377,...,70.300000,-0.02,2260.08,8024.0,9.19,2.22,92.0,1.87,1.49,2.755964e+07
4,-2.18,5.83,6.16,4.23,18.0,0.46,25.04,24.338873,23.401887,1.916853,...,70.700000,0.02,457.68,-7687.0,9.06,2.24,94.0,1.81,1.51,4.480113e+07
5,1.92,6.09,6.20,4.16,18.0,0.48,23.36,23.628342,24.176307,-1.990291,...,71.300000,0.01,-29.53,-5657.0,9.06,2.21,90.0,1.82,1.60,4.073372e+07
6,-1.83,6.10,6.22,4.13,18.0,0.46,23.30,22.657828,24.635744,-1.434902,...,71.000000,-0.01,1090.54,770.0,9.03,2.18,89.0,1.84,1.57,6.152815e+07
7,6.42,5.87,6.17,4.26,18.0,0.41,23.06,22.169002,24.631415,-0.888802,...,71.200000,-0.01,82.22,-4059.0,9.03,2.03,89.0,1.88,1.54,3.940499e+07
8,7.54,5.78,6.05,4.22,18.0,0.33,23.30,21.905686,24.522817,0.136128,...,70.700000,0.01,884.41,-3477.0,9.03,1.84,90.0,1.83,1.54,1.687391e+07


In [5]:
discretised_fi = pd.DataFrame(index=df_fi.index)

for col in df_fi.columns:
    no_unique = df_fi[col].nunique()
    
    if no_unique <= 1:
        discretised_fi[col] = 0
    else:
        try:
            discretised_fi[col] = pd.qcut(
                df_fi[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_fi[col] = df_fi[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_fi)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    2    0    2     2     0    0     2     2     0      2  ...    2    0   
1    2    1    2     2     0    2     2     2     0      1  ...    2    2   
2    0    2    2     2     0    2     1     0     1      0  ...    2    0   
3    1    1    0     1     0    1     2     1     1      2  ...    0    0   
4    0    0    0     1     0    1     1     2     0      2  ...    0    2   
5    1    2    1     0     0    2     1     1     1      0  ...    1    2   
6    0    2    1     0     0    1     0     1     2      0  ...    1    1   
7    1    1    1     1     0    0     0     0     2      1  ...    1    1   
8    2    0    0     0     0    0     0     0     2      1  ...    0    1   

   EPFRG  GGAG  DI  GE  CPI  RQ  VAA  EUPC  
0      0     2   0   0    2   0    0     0  
1      1     0   1   1    1   0    0     0  
2      0     1   0   2    0   0    0     1  
3      2     2   1   2    2   2

In [6]:
sm = from_pandas(discretised_fi)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_fi.html")

Graphs/01_fully_connected_fi.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_fi.html")

Graphs/01_thresholded_fi.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_fi_1.html")

In [10]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

    From     To    Weight
39  LBGD    EMP  2.529839
57    DI    LFG  2.311511
58    DI  EPFRG  2.274660
54    DI   GDPC  2.151197
13  VABI    EMP  2.097153
..   ...    ...       ...
15  VABI  EPFRG -2.097788
0    FDI   EUPC -2.253376
33  EBSG    EMP -2.329760
12  VABI   GDPC -2.483938
20  GCFP    EMP -2.839176

[67 rows x 3 columns]


In [12]:
df_edges.to_csv("fi_2007_2015.csv")